In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression

PROBLEM 1: Data analysis using markov chians 

In this problem, you will empirically analyze a Markov chain 
with a finite state space. Transition probabilities are unknown.

The state space is:
    S = {0, 1, 2, 3}

You are given the data for the observed X_t for t  = 0..19

Tasks:
1. Estimate the transition matrix P from the observed transitions.
2. Verify that the estimated matrix is a probability transition matrix.
3. Compute the stationary distribution pi of the chain.
4. Simulate the chain using the estimated transition matrix
5. Compute the expected hitting times via

   (a) Simulation

   (b) Solving linear equations (analytical hitting times). 

Compare the estimates and interpret the results


In [23]:
import numpy as np

# state space
S = [0, 1, 2, 3]
N_states = len(S)

# Observed transitions: each row is (current_state, next_state)
X_t = np.array([
    [0, 1],
    [1, 2],
    [2, 3],
    [3, 0],
    [0, 1],
    [1, 1],
    [1, 2],
    [2, 2],
    [2, 3],
    [3, 3],
    [3, 0],
    [0, 2],
    [2, 1],
    [1, 3],
    [3, 1],
    [1, 0],
    [0, 0],
    [0, 1],
    [1, 2],
    [2, 0],
], dtype=int)




Below are methods that you need to complete

In [24]:
# 1.1
def comp_transition_matrix(transitions, n_states):
    """
    Estimate the transition matrix P from observed transitions.

    Args:
        transitions: array of shape (n_samples, 2)
        n_states: number of states

    Returns:
        P_hat: estimated transition matrix
    """
    P_hat = np.zeros((n_states, n_states))
    
    n_transitions = transitions.shape[0]
    for t in transitions:
        
        P_hat[t[0],t[1]] += 1

    row_sums = P_hat.sum(axis=1)
    P_hat = P_hat / row_sums[:, np.newaxis]

    return P_hat

P = comp_transition_matrix(X_t, N_states)
print(P)

#  1.2
def is_transition_matrix(P):
    """
    Check if P is a transition matrix.
    """

    if P.ndim != 2 or P.shape[0] != P.shape[1]:
        print("Wrong shape")
        return False

    if not np.isfinite(P).all():
        print("Is not finite")
        return False

    if np.all(P < 0):
        print("Has negative elements")
        return False

    if not np.allclose(P.sum(axis=1), 1.0, atol=1e-8):
        print("Rows do not sum to 1")
        return False
    
    return True

print(is_transition_matrix(P))

# 1.3
def stationary_distribution(P):
    """
    Compute stationary distribution
    """

    P = np.array(P)
    eigenvals, eigenvecs = np.linalg.eig(P.T)

    # find the index of the eigenvalue that is 1
    index = np.argmin(np.abs(eigenvals - 1))

    stat = eigenvecs[:, index].flatten() # getting eigenvec with eigenval 1
    stat = stat / np.sum(stat) #normalizing vector
    return stat

print(stationary_distribution(P))



def simulate_chain(P, start_state, n_steps):
    """
    Simulate a Markov chain trajectory with a fixed random seed.

    Returns: array of visited states of length n_steps + 1
    """
    # seed = 1234  # don't change that
    rng = np.random.default_rng()

    path = np.zeros(n_steps + 1, dtype=int)
    path[0] = start_state

    for t in range(n_steps):
        path[t + 1] = rng.choice(
            P.shape[0],
            p=P[path[t]]
        )

    return path

print(simulate_chain(P, 0, 4))



def hitting_times_sim(P, start_state, n_sim=10_000):
    """
    Estimate expected hitting times E[T_{start -> j}] for ALL states j.

    Returns:
        est: 1D array, where est[j] the estimated expected steps to hit state j from start_state. 
    """
    
    seed = 1234
    result = []

    # Find simulation estimates of hitting time for all states 0,1, 2, 3
    for state in range(0,P.shape[0]):
        list_steps = []
        for _ in range(n_sim):
            current_state = start_state
            steps = 0
            while current_state != state:
                current_state = simulate_chain(P, current_state, n_steps=1)[-1]
                steps += 1

            list_steps.append(steps)
        result.append(np.mean(list_steps)+1)

    return result

print(hitting_times_sim(P, start_state=0))


def theoretical_hitting_times(P, start_state):
    N_states = P.shape[0]
    hit_theor = np.full(N_states, np.nan, dtype=float)

    I = np.eye(N_states)
    ones = np.ones(N_states)

    for target in range(N_states):
        A = I - P
        b = ones.copy()

        # impose boundary condition h_target = 1
        A[target, :] = 0.0
        A[target, target] = 1.0
        b[target] = 1.0

        try:
            h = np.linalg.solve(A, b)
            hit_theor[target] = h[start_state]
        except np.linalg.LinAlgError:
            # singular system → target not reachable
            hit_theor[target] = np.inf

    return hit_theor

print(theoretical_hitting_times(P, start_state=0))

[[0.2        0.6        0.2        0.        ]
 [0.16666667 0.16666667 0.5        0.16666667]
 [0.2        0.2        0.2        0.4       ]
 [0.5        0.25       0.         0.25      ]]
True
[0.25-0.j 0.3 -0.j 0.25-0.j 0.2 -0.j]
[0 1 1 2 3]
[np.float64(1.0), np.float64(3.0319), np.float64(4.3093), np.float64(6.7126)]
[1.         3.02439024 4.31707317 6.68292683]


When you are done, run the following cell (no need to implement anything else)

In [25]:
def problem1_main():
    print("\n=== Problem 1: Markov chain estimation + hitting times ===")

    # 1) Estimate P
    P_hat = comp_transition_matrix(X_t, N_states)
    print("Estimated P_hat:\n", np.round(P_hat, 3))

    # 2) Validate
    print("Is valid transition matrix?", is_transition_matrix(P_hat))

    # 3) Expected steps from given start state to all states
    start_state = 0

    # simulation
    mc = hitting_times_sim(P_hat, start_state=start_state, n_sim=5000)

    # Theory (linear system)
    th = theoretical_hitting_times(P_hat, start_state=start_state)

    # 4) Compare
    df = pd.DataFrame({
        "target_state": np.arange(N_states),
        "MC_estimate": mc,
        "theoretical": th,
        "abs_diff": np.abs(mc - th),
    })
    print("\nComparison table:\n", df)

problem1_main()


=== Problem 1: Markov chain estimation + hitting times ===
Estimated P_hat:
 [[0.2   0.6   0.2   0.   ]
 [0.167 0.167 0.5   0.167]
 [0.2   0.2   0.2   0.4  ]
 [0.5   0.25  0.    0.25 ]]
Is valid transition matrix? True

Comparison table:
    target_state  MC_estimate  theoretical  abs_diff
0             0       1.0000     1.000000  0.000000
1             1       3.0480     3.024390  0.023610
2             2       4.3506     4.317073  0.033527
3             3       6.7038     6.682927  0.020873


PROBLEM 2: Cost-Sensitive Classification

You are given a binary classification problem for fraud detection.

Class labels:

    y = 1 => fraud

    y = 0 => ok



The costs of classification outcomes are:
    TP = 0, TN = 0, FP = 100, FN = 500

Tasks:
1. Train an SVM classifier.
2. Compute classification costs at a fixed threshold (0.5).
3. Evaluate total cost for multiple probability thresholds.
4. Find the threshold that minimizes total cost.

In [26]:
import numpy as np
import pandas as pd

costs = {"TP": 0, "TN": 0, "FP": 100, "FN": 500}


def generate_fraud_table(seed=0, n=3000, fraud_rate=0.05):
    """
    Generate a simple fraud dataset as a single table. The table contains:
        - numerical features: x1, x2, x3
        - binary target column: fraud (1 = fraud, 0 = legitimate)
    """
    rng = np.random.default_rng(seed)

    # Target variable
    fraud = (rng.random(n) < fraud_rate).astype(int)

    # Features
    x1 = rng.normal(0, 1, size=n)
    x2 = rng.normal(0, 1, size=n)
    x3 = rng.normal(0, 1, size=n)

    #  fraud cases are shifted
    x1[fraud == 1] += 2.0
    x2[fraud == 1] += 1.0

    df = pd.DataFrame({
        "x1": x1,
        "x2": x2,
        "x3": x3,
        "fraud": fraud,
    })

    return df


fraud_data = generate_fraud_table()

fraud_data.head()

,x1,x2,x3,fraud
0,-0.250243,-0.863902,-0.307019,0
1,-0.380736,0.018756,-0.559577,0
2,1.126431,2.055912,0.973126,1
3,0.806991,2.104160,-0.211368,1
4,0.059649,0.652374,-0.437259,0


Fill in the methods in the cell below:

In [ ]:
#from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split


def train_test_split_table(df):
    """
    Split a data table into training and test sets.

    Returns:
        X_train, X_test, y_train, y_test
    """
    # implement splitting
    # first, decide what are features and what are target 
    X = df[['x1', 'x2', 'x3']]
    y = df['fraud']

    # then split into train and test
    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, shuffle=True)

    return X_train, X_test, y_train, y_test

def fit_linear_svm(fraud_data):
    """
    Fit a linear SVM classifier.

    Args: data table

    Returns:
        predicted labels of length len(y_test) 
    """
    # define our model
    clf = LinearSVC(
        C=1.0,
        max_iter=10_000,
        random_state=0
    )

    # split the data into trian and test:
    X_train, X_test, y_train, y_test = train_test_split_table(fraud_data)
    #   Fit the SVM using X_train and y_train and predict the label using y_test. return y_pred
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    return clf



def confusion_counts(y_true, y_pred):
    
    """
    Computes TP, TN, FP, FN.
    """
    from sklearn.metrics import confusion_matrix


    TN_est, FP_est, FN_est, TP_est = confusion_matrix(y_true, y_pred).ravel().tolist()
    
    # Here you Ccmpute TP, TN, FP, FN.
    
    return {"TP": TP_est, "TN": TN_est, "FP": FP_est, "FN": FN_est}


def total_cost(counts):
    """
    Compute total cost from confusion counts.

    """
    # Multiply counts by costs and sum
    total_cost = counts["FP"]*100 + counts["FN"]*500
    
    return total_cost

# evaluate how the classification cost changes when you change the decision threshold.
def sweep_thresholds(y_true, thresholds, X, clf):
    """
    Evaluate total cost for a range of thresholds.
    
    Here, clf is your trained SVM classifier
    """

    results = []
    
    # note: here, I define y_probs to be just a decision function. Think: does it need to be calibrated to be used in this problem?
    y_probs = clf.decision_function(X)

    for t in thresholds:
        # 1) compute the prediction for a chosen theshold
        y_pred = (y_probs >= t).astype(int)

        # 2) Confusion matrix counts  (previoulsy implemented by you)
        counts = confusion_counts(y_true, y_pred)

        # 3) Total cost (previoulsly implemented by you)
        cost = total_cost(counts)

        # 4) Store results
        results.append({
            "threshold": t,
            "TP": counts["TP"],
            "TN": counts["TN"],
            "FP": counts["FP"],
            "FN": counts["FN"],
            "total_cost": cost,
        })

    return pd.DataFrame(results)



When you are done, run the following cell (no need to implement anything else)

In [28]:
def main():

    df = fraud_data

    print("Dataset head:")
    print(df.head(), "\n")

    # split in train and test:
    _, X_test, _, y_test = train_test_split_table(df)
    # Fit linear SVM
    clf = fit_linear_svm(df)

    # thresholds
    thresholds = np.linspace(-2.0, 2.0, 21)
    df_results = sweep_thresholds(
        y_test,
        thresholds,
        X_test,
        clf,
    )

    print("Threshold sweep results:")
    print(df_results)

    # 6) Identify optimal threshold
    best_row = df_results.loc[df_results["total_cost"].idxmin()]
    print("Optimal threshold:", best_row)

main()

Dataset head:
         x1        x2        x3  fraud
0 -0.250243 -0.863902 -0.307019      0
1 -0.380736  0.018756 -0.559577      0
2  1.126431  2.055912  0.973126      1
3  0.806991  2.104160 -0.211368      1
4  0.059649  0.652374 -0.437259      0 

Threshold sweep results:
    threshold  TP   TN   FP  FN  total_cost
0        -2.0  36  168  396   0       39600
1        -1.8  36  239  325   0       32500
2        -1.6  36  301  263   0       26300
3        -1.4  36  375  189   0       18900
4        -1.2  35  422  142   1       14700
5        -1.0  34  463  101   2       11100
6        -0.8  33  498   66   3        8100
7        -0.6  31  528   36   5        6100
8        -0.4  27  544   20   9        6500
9        -0.2  22  557    7  14        7700
10        0.0  20  562    2  16        8200
11        0.2  17  564    0  19        9500
12        0.4  11  564    0  25       12500
13        0.6   7  564    0  29       14500
14        0.8   4  564    0  32       16000
15        1.0   3  56

PROBLEM 3: Confidence estimation of the cost

In Problem 2, you trained a classifier, selected a decision threshold, evaluated its performance on a test set, and computed the cost

In this problem, you will quantify the uncertainty of this estimated cost. Each observation in the test set produces a cost depending on the
classification outcome:

    TN: 0
   
    FP: 100

    TP: 0

    FN: 500

Thus, the cost per observation is a bounded random variable taking
values in the interval [0, 500].

Tasks:
1. Compute the average cost per observation on the test set.
2. Use Hoeffding’s inequality to construct a 95% confidence interval
   for the true expected cost of the classifier.
3. Interpret the resulting interval:
   - What does it say about the reliability of your estimate?
   - Is the interval likely to be tight or conservative? Why?

You may assume that test observations are independent and identically
distributed.

In [33]:
def per_observation_cost(y_true, y_pred):
    """
    Compute per-observation cost vector.
    """
    
    # here, you will compute the average cost using the test set
    true_pred = np.column_stack([y_true, y_pred])
    cost_seq = []

    for row in true_pred:
        if list(row) == [0, 1]:
            cost_seq.append(100)
        elif list(row) == [1, 0]:
            cost_seq.append(500)
        else:
            cost_seq.append(0)
    
    return np.array(cost_seq)


def hoeffding_ci(per_obs_costs, a, b, delta=0.05):
    """
    Hoeffding confidence interval
    """
    # Step 1: deterministic costs per observation
    c = per_obs_costs

    # Step 2:   average cost
    mean_cost = np.mean(c)
    n = len(c)

    epsilon = (b - a) / np.sqrt(2 * n) * np.sqrt(np.log(2 / delta))
    # Step 3: construct a Hoeffding intevral of the estimated cost
    ci = (mean_cost - epsilon, mean_cost + epsilon)

    return ci

In [35]:
X_train, X_test, y_train, y_test = train_test_split_table(fraud_data)

clf = LinearSVC(
        C=1.0,
        max_iter=10_000,
        random_state=0
    )
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

seq = per_observation_cost(y_test, y_pred)
print(np.mean(seq))

print(hoeffding_ci(seq, a=0, b=500))

18.333333333333332
(np.float64(-9.388797770541114), np.float64(46.055464437207775))
